# Chapter 21
## Gap Junctions
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np

### Figure 21.1
Choosing a Reset and Threshold for the WB Neuron's LIF Approximation

Reuses the WB neuron from chapter 5 (`m` at its instantaneous
equilibrium). The dashed lines at -67 and -52 mV mark the reset/threshold
values used to approximate this neuron as a LIF neuron for the network
sub-examples below.

In [ ]:
def simulate_WB_neuron(input_current, simulation_time, dt=0.01 * b2.ms):
    El = -65 * b2.mV
    EK = -90 * b2.mV
    ENa = 55 * b2.mV
    gl = 0.1 * b2.msiemens
    gK = 9 * b2.msiemens
    gNa = 35 * b2.msiemens
    C = 1 * b2.ufarad

    eqs = """
    I_e : amp

    alphah = 0.35 * exp(-(vm + 58.0*mV) / (20.0*mV))/ms :Hz
    alpham = 0.1/mV * (vm + 35.0*mV) / (1.0 - exp(-0.1/mV * (vm + 35.0*mV))) /ms :Hz
    alphan = -0.05/mV * (vm + 34.0*mV) / (exp(-0.1/mV * (vm + 34.0*mV)) - 1.0)/ms :Hz

    betah = 5.0 / (exp(-0.1/mV * (vm + 28.0*mV)) + 1.0)/ms :Hz
    betam = 4.0 * exp(-(vm + 60.0*mV) / (18.0*mV))/ms :Hz
    betan = 0.625 * exp(-(vm + 44.0*mV) / (80.0*mV))/ms :Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + gNa*m**3*h*(ENa-vm) + \
        gl*(El-vm) + gK*n**4*(EK-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dvm/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(1, eqs, method="rk4", dt=dt)
    neuron.vm = -63 * b2.mV
    neuron.I_e = input_current
    neuron.h = "alphah / (alphah + betah)"
    neuron.n = "alphan / (alphan + betan)"

    st_mon = b2.StateMonitor(neuron, ["vm"], record=True)
    net = b2.Network(neuron)
    net.add(st_mon)
    net.run(simulation_time)
    return st_mon


sm_rt = simulate_WB_neuron(0.75 * b2.uA, 100 * b2.ms)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sm_rt.t / b2.ms, sm_rt.vm[0] / b2.mV, "-k", lw=2)
ax.plot([0, 100], [-67, -67], "--b", lw=1)
ax.plot([0, 100], [-52, -52], "--b", lw=1)
ax.set_xlim(0, 100)
ax.set_ylim(-100, 50)
ax.set_xlabel("t [ms]")
ax.set_ylabel("v [mV]")
plt.tight_layout()
plt.show()

### Two LIF Neurons Coupled by a Gap Junction

Dimensionless LIF model (same convention as chapter 7), coupled by a
diffusive gap-junction term `g_gap*(v_other - v_self)`. On top of the
continuous gap current, a spike also delivers an instantaneous
`epsilon` kick to the other cell -- modeled here with a synapse that
fires only on a presynaptic spike (`on_pre`), separate from the
always-on gap-junction current (modeled as a summed synaptic
variable).

In [ ]:
def simulate_LIF_gap_junction(i_ext, g_gap, epsilon, simulation_time, dt=0.002 * b2.ms):
    eqs = """
    dv/dt = (-v/tau + i_ext + Igap)/ms : 1
    Igap : 1
    i_ext : 1
    """
    tau = 10.0
    neurons = b2.NeuronGroup(2, eqs, threshold="v>1", reset="v=0",
                             method="euler", dt=dt, namespace={"tau": tau})
    neurons.v = [0.4, 0.9]
    neurons.i_ext = i_ext

    gap = b2.Synapses(neurons, neurons, "w : 1\nIgap_post = w*(v_pre - v_post) : 1 (summed)")
    gap.connect(condition="i!=j")
    gap.w = g_gap

    on_pre = """
    v_post += epsilon
    v_post = v_post * int(v_post <= 1)
    """
    kick = b2.Synapses(neurons, neurons, on_pre=on_pre, namespace={"epsilon": epsilon})
    kick.connect(condition="i!=j")

    st_mon = b2.StateMonitor(neurons, "v", record=True)
    sp_mon = b2.SpikeMonitor(neurons)
    net = b2.Network(neurons, gap, kick, st_mon, sp_mon)
    net.run(simulation_time)
    return st_mon, sp_mon

### Figure 21.2
Two Coupled LIF Neurons: Coupled (black) vs. Uncoupled (red)

In [ ]:
i_ext = np.array([0.125, 0.09])
g_gap, beta = 0.01, 3.0

sm_c, spm_c = simulate_LIF_gap_junction(i_ext, g_gap, beta * g_gap, 100 * b2.ms)
sm_u, spm_u = simulate_LIF_gap_junction(i_ext, g_gap, 0.0, 100 * b2.ms)

fig, ax = plt.subplots(2, figsize=(7, 6))
for k in range(2):
    for st in spm_c.t[spm_c.i == k] / b2.ms:
        ax[k].plot([st, st], [0, 6], "-k", lw=3)
    for st in spm_u.t[spm_u.i == k] / b2.ms:
        ax[k].plot([st, st], [0, 6], "-r", lw=1)
    ax[k].plot(sm_c.t / b2.ms, sm_c.v[k], "-k", lw=3)
    ax[k].plot(sm_u.t / b2.ms, sm_u.v[k], "-r", lw=1)
    ax[k].set_xlim(0, 100)
    ax[k].set_ylabel(f"$v_{k+1}$ [mV]")
ax[0].set_ylim(0, 6)
ax[1].set_ylim(0.85, 0.95)
ax[1].set_xlabel("t [ms]")
plt.tight_layout()
plt.show()

### Two WB Neurons Coupled by a Gap Junction

In [ ]:
def simulate_WB_gap_junction(i_ext, gap_strength, simulation_time, dt=0.01 * b2.ms,
                              rectifying=False, thr=-50 * b2.mV):
    El = -65 * b2.mV
    EK = -90 * b2.mV
    ENa = 55 * b2.mV
    gl = 0.1 * b2.msiemens
    gK = 9 * b2.msiemens
    gNa = 35 * b2.msiemens
    C = 1 * b2.ufarad

    eqs = """
    I_e : amp
    Igap_a : amp
    Igap_b : amp
    Igap = Igap_a + Igap_b : amp

    alphah = 0.35 * exp(-(vm + 58.0*mV) / (20.0*mV))/ms :Hz
    alpham = 0.1/mV * (vm + 35.0*mV) / (1.0 - exp(-0.1/mV * (vm + 35.0*mV))) /ms :Hz
    alphan = -0.05/mV * (vm + 34.0*mV) / (exp(-0.1/mV * (vm + 34.0*mV)) - 1.0)/ms :Hz

    betah = 5.0 / (exp(-0.1/mV * (vm + 28.0*mV)) + 1.0)/ms :Hz
    betam = 4.0 * exp(-(vm + 60.0*mV) / (18.0*mV))/ms :Hz
    betan = 0.625 * exp(-(vm + 44.0*mV) / (80.0*mV))/ms :Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + Igap + gNa*m**3*h*(ENa-vm) + \
        gl*(El-vm) + gK*n**4*(EK-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dvm/dt = membrane_Im/C : volt
    """

    neurons = b2.NeuronGroup(2, eqs, method="rk4", dt=dt)
    neurons.vm = -63 * b2.mV
    neurons.I_e = i_ext
    neurons.h = "alphah / (alphah + betah)"
    neurons.n = "alphan / (alphan + betan)"

    if not rectifying:
        gap = b2.Synapses(neurons, neurons,
                           "w : siemens\nIgap_a_post = w*(vm_pre - vm_post) : amp (summed)")
        gap.connect(condition="i!=j")
        gap.w = gap_strength
        objects = [gap]
    else:
        # coupling is gated by neuron 0's voltage alone (as in the book's
        # script), not by each synapse's own presynaptic voltage. Two
        # separate Synapses (rather than one connect(condition="i!=j"))
        # so each can reference the right one of vm_pre/vm_post for
        # "neuron 0's voltage", and two different summed targets
        # (Igap_a, Igap_b) since Brian2 forbids two Synapses summing
        # into the same target variable.
        gap_out = b2.Synapses(neurons, neurons,
                               "w : siemens\nIgap_a_post = w*int(vm_pre<=thr)*(vm_pre-vm_post) : amp (summed)",
                               namespace={"thr": thr})
        gap_out.connect(i=0, j=1)
        gap_out.w = gap_strength
        gap_in = b2.Synapses(neurons, neurons,
                              "w : siemens\nIgap_b_post = w*int(vm_post<=thr)*(vm_pre-vm_post) : amp (summed)",
                              namespace={"thr": thr})
        gap_in.connect(i=1, j=0)
        gap_in.w = gap_strength
        objects = [gap_out, gap_in]

    st_mon = b2.StateMonitor(neurons, "vm", record=True)
    net = b2.Network(neurons, *objects, st_mon)
    net.run(simulation_time)
    return st_mon

### Figure 21.3
Two WB Neurons Coupled by an Always-On Gap Junction

In [ ]:
sm_wb = simulate_WB_gap_junction([1.0, 0.0] * b2.uA, 0.01 * b2.msiemens, 200 * b2.ms)

fig, ax = plt.subplots(2, figsize=(7, 5), sharex=True)
ax[0].plot(sm_wb.t / b2.ms, sm_wb.vm[0] / b2.mV, lw=2, c="k")
ax[1].plot(sm_wb.t / b2.ms, sm_wb.vm[1] / b2.mV, lw=2, c="k")
for i in range(2):
    ax[i].set_xlim(100, 200)
    ax[i].set_xlabel("time [ms]")
ax[0].set_ylabel("v1 [mV]")
ax[1].set_ylabel("v2 [mV]")
ax[0].set_ylim(-100, 50)
ax[1].set_ylim(-64, -62)
plt.tight_layout()
plt.show()

### Figure 21.4
Two WB Neurons Coupled by a Rectifying (Subthreshold-Only) Gap Junction

Coupling is on only while neuron 1's voltage stays at or below -50mV
(black: always-on gap junction; red: subthreshold-only gap junction).

In [ ]:
sm_always = simulate_WB_gap_junction([1.0, 0.0] * b2.uA, 0.01 * b2.msiemens, 100 * b2.ms)
sm_sub = simulate_WB_gap_junction([1.0, 0.0] * b2.uA, 0.01 * b2.msiemens, 100 * b2.ms,
                                   rectifying=True, thr=-50 * b2.mV)

fig, ax = plt.subplots(2, figsize=(7, 6))
ax[0].plot(sm_always.t / b2.ms, sm_always.vm[0] / b2.mV, "-k", lw=3)
ax[0].plot(sm_sub.t / b2.ms, sm_sub.vm[0] / b2.mV, "-r", lw=1)
ax[0].set_ylabel("$v_1$ [mV]")
ax[0].set_xlim(0, 100)
ax[0].set_ylim(-100, 50)

ax[1].plot(sm_always.t / b2.ms, sm_always.vm[1] / b2.mV, "-k", lw=3)
ax[1].plot(sm_sub.t / b2.ms, sm_sub.vm[1] / b2.mV, "-r", lw=1)
ax[1].set_xlabel("t [ms]")
ax[1].set_ylabel("$v_2$ [mV]")
ax[1].set_xlim(0, 100)
tail = sm_always.vm[1][len(sm_always.t) // 2:] / b2.mV
ax[1].set_ylim(tail.min() - 0.75, tail.max() + 0.75)
plt.tight_layout()
plt.show()